In [2]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0
%pylab inline
%load_ext autoreload
%autoreload 2

env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

/gpfs01/berens/user/pgivehchian/ssl_neuron/.venv39/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from ssl_neuron.datasets import GraphDataset
from ssl_neuron.utils import plot_neuron, neighbors_to_adjacency_torch, compute_eig_lapl_torch_batch
from ssl_neuron.graphdino import create_model

In [5]:
TRIAL = "trial5"

#### Load config

In [6]:
config = json.load(open(f'../configs/{TRIAL}.json'))

#### Load model + checkpoint

In [7]:
cell_meta = pd.read_csv(os.path.join(config['data']['path'], 'cell_meta.csv'))
cell_meta['cell_id'] = cell_meta['cell_id'].astype(str)

In [8]:
cell_meta

,cell_id,soma_x,soma_y,soma_z,n_nodes_raw,n_nodes,n_axon_nodes,n_stitched_edges,cellclass_final,valid_cellclass_final,valid_status,celltype_final,valid_celltype_final,celltype_final_decision,soma_x_um,soma_y_um,soma_z_um
0,720575940537162675,623.088321,271.935098,-21.628196,9054,8096,958,0,RGC,True,True,NaN,False,NaN,623.088,271.935,-21.6282
1,720575940537274291,483.576538,891.599123,-14.109626,5112,3943,1169,0,RGC,True,True,ON alpha,True,both_strong,483.577,891.599,-14.1096
2,720575940537284275,477.293864,607.582202,-18.812998,6950,6137,813,0,RGC,True,True,Sustained suppressed-by-contrast strong surrou...,True,both_weak,477.294,607.582,-18.8130
3,720575940537313971,778.958349,719.019178,-14.450323,8642,7627,1015,0,RGC,True,True,ON delayed,True,both_strong,778.958,719.019,-14.4503
4,720575940537343155,283.318443,409.646856,-16.305329,9935,8934,1001,0,RGC,True,True,OFF vertical OS - asymmetric,True,both_strong,283.318,409.647,-16.3053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3685,720575940589082516,271.145212,1050.133062,-13.881681,9780,9657,123,0,RGC,True,True,NaN,False,NaN,271.145,1050.130,-13.8817
3686,720575940589088148,281.992055,1067.835937,-14.430131,17577,14828,2749,0,RGC,True,True,NaN,False,NaN,281.992,1067.840,-14.4301
3687,720575940589111700,851.058152,1070.755962,-14.051774,13661,11003,2658,0,RGC,True,True,HD2,True,both_strong,851.058,1070.760,-14.0518
3688,720575940589184404,1036.643098,993.958841,-13.925180,16997,15377,1620,0,RGC,True,True,HD1,True,both_weak,1036.640,993.959,-13.9252


In [9]:
model = create_model(config)

In [11]:
state_dict = torch.load(f'../ckpts/{TRIAL}.pt')

In [12]:
model.load_state_dict(state_dict)

<All keys matched successfully>

In [13]:
model.eval()
model.cuda();

#### Load dataset

In [14]:
dset = GraphDataset(config, mode='all')

100%|██████████| 3690/3690 [04:59<00:00, 12.32it/s]


#### Run inference

In [18]:
latents = np.zeros((dset.num_samples, config['model']['dim']))

cell_meta['latent'] = pd.Series(
    [None] * len(cell_meta),
    index=cell_meta.index,
    dtype=object
)

cell_meta['cell_id'] = cell_meta['cell_id'].astype(str)

for i in tqdm(range(dset.num_samples)):
    feat, neigh, cell_id = dset.__getsingleitem__(i)
    cell_id = str(cell_id)

    adj = neighbors_to_adjacency_torch(
        neigh, list(neigh.keys())
    ).float().cuda()[None, :]

    lapl = compute_eig_lapl_torch_batch(
        adj,
        pos_enc_dim=config['model']['pos_dim']
    ).float().cuda()

    feat = torch.from_numpy(feat).float().cuda()[None, :]

    with torch.no_grad():
        embedding = model.student_encoder.forward(feat, adj, lapl)[0]
        embedding = embedding.cpu().numpy().squeeze()

    latents[i] = embedding

    mask = cell_meta['cell_id'] == cell_id

    for idx in cell_meta.index[mask]:
        cell_meta.at[idx, 'latent'] = embedding

print("\nEmbedding extraction complete!")
print("Latents matrix shape:", latents.shape)
print("Rows with embeddings:", cell_meta['latent'].notna().sum())


100%|██████████| 3690/3690 [04:02<00:00, 15.24it/s]


Embedding extraction complete!
Latents matrix shape: (3690, 32)
Rows with embeddings: 3690


In [19]:
z_means = []

for _, row in tqdm(cell_meta.iterrows(), total=len(cell_meta)):

    cell_id = int(row['cell_id'])

    feature_path = os.path.join(
        config['data']['path'],
        'skeletons',
        str(cell_id),
        'features.npy'
    )

    if os.path.exists(feature_path):

        features = np.load(feature_path)
        z_values = features[:, 2]
        z_mean = np.nanmean(z_values)

    else:
        print(f"Missing features: {cell_id}")
        z_mean = np.nan

    z_means.append(z_mean)

cell_meta['z_mean'] = z_means

100%|██████████| 3690/3690 [00:02<00:00, 1568.21it/s]


In [20]:
cell_meta.to_csv(
    os.path.join(config['trainer']['latents_dir'], f'{TRIAL}.csv'),
    index=False
)